# NI USB-6002 Data Capture + Spectral Analysis (Simple)

Run cells top-to-bottom.

In [ ]:
# If you see 'ModuleNotFoundError: nidaqmx', install it in your environment:
#   pip install nidaqmx
#
# If spectrogram/peaks fail due to missing SciPy:
#   conda install -c conda-forge scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nidaqmx
from nidaqmx.constants import TerminalConfiguration, AcquisitionType

try:
    from scipy.signal import find_peaks, spectrogram, windows
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False


In [ ]:
# ---- SETTINGS (edit these) ----
DEVICE = "Dev1"          # e.g., "Dev1"
CHANNEL = "ai0"          # e.g., "ai0" (single channel)
PHYS_CHAN = f"{DEVICE}/{CHANNEL}"

FS_HZ = 1000.0           # sample rate (Hz)
N_SAMPLES = 10_000       # total samples to read

# Input range (Volts)
MIN_V = -10.0
MAX_V =  10.0

# Terminal configuration (choose ONE)
# TERM_CFG = TerminalConfiguration.RSE
TERM_CFG = TerminalConfiguration.DIFFERENTIAL

# CSV output
CSV_PATH = "ni_usb6002_capture.csv"


In [ ]:
# ---- ACQUIRE DATA (finite capture) ----
# Reads exactly N_SAMPLES at FS_HZ from PHYS_CHAN.

dt = 1.0 / FS_HZ
t = np.arange(N_SAMPLES) * dt

with nidaqmx.Task() as task:
    try:
        task.ai_channels.add_ai_voltage_chan(
            PHYS_CHAN,
            min_val=MIN_V,
            max_val=MAX_V,
            terminal_config=TERM_CFG,
        )
    except Exception as e:
        raise RuntimeError(
            f"Could not add channel '{PHYS_CHAN}'.\n"
            f"- Check DEVICE/CHANNEL names (NI MAX is the source of truth)\n"
            f"- Make sure the NI-DAQmx driver is installed\n"
            f"- Confirm the USB-6002 is connected and visible in NI MAX\n\n"
            f"Original error: {e}"
        )

    task.timing.cfg_samp_clk_timing(
        rate=FS_HZ,
        sample_mode=AcquisitionType.FINITE,
        samps_per_chan=N_SAMPLES,
    )

    # Read returns a Python list of length N_SAMPLES for a single channel
    y = np.array(task.read(number_of_samples_per_channel=N_SAMPLES, timeout=10.0), dtype=float)

print(f"Captured {len(y)} samples at {FS_HZ:.1f} Hz ({len(y)/FS_HZ:.2f} s) from {PHYS_CHAN}.")


In [ ]:
# ---- PLOT TIME SERIES + EXPORT CSV ----
df = pd.DataFrame({
    "t_s": t,
    "V": y,
})

df.to_csv(CSV_PATH, index=False)
print(f"Saved CSV: {CSV_PATH}")

plt.figure()
plt.plot(df["t_s"], df["V"])
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Time Series")
plt.grid(True)
plt.show()


In [ ]:
# ---- SPECTRUM (FFT) + PEAKS ----
# Uses a Hann window for cleaner peaks. Reports peak frequencies and amplitudes.

N = len(y)
y_detrend = y - np.mean(y)

if _HAVE_SCIPY:
    w = windows.hann(N, sym=False)
else:
    w = np.hanning(N)

yw = y_detrend * w

# FFT (one-sided)
Y = np.fft.rfft(yw)
f = np.fft.rfftfreq(N, d=1.0/FS_HZ)

# Amplitude scaling (approx. volts peak for a sinusoid)
cg = np.mean(w)  # coherent gain
A = (2.0 / (N * cg)) * np.abs(Y)
A[0] = A[0] / 2.0  # DC shouldn't be doubled

# Peak picking
if _HAVE_SCIPY:
    peaks, _ = find_peaks(A, height=np.max(A)*0.05)  # peaks >= 5% of max
else:
    thresh = np.max(A)*0.05
    peaks = np.where((A[1:-1] > A[:-2]) & (A[1:-1] > A[2:]) & (A[1:-1] >= thresh))[0] + 1

peaks = peaks[np.argsort(A[peaks])[::-1]]  # sort by amplitude (desc)

TOP_K = 8
print("Top peaks (frequency Hz, amplitude V_peak approx.):")
for i, p in enumerate(peaks[:TOP_K], start=1):
    print(f"{i:>2}. {f[p]:>10.3f} Hz   {A[p]:>10.6f} V")

plt.figure()
plt.plot(f, A)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude (V_peak, approx.)")
plt.title("Amplitude Spectrum (FFT)")
plt.grid(True)
plt.show()


In [ ]:
# ---- SPECTROGRAM ----
if not _HAVE_SCIPY:
    raise RuntimeError("SciPy is required for the spectrogram cell. Install with: conda install -c conda-forge scipy")

nperseg = 1024
noverlap = nperseg // 2

f_s, t_s, Sxx = spectrogram(
    y_detrend,
    fs=FS_HZ,
    window="hann",
    nperseg=nperseg,
    noverlap=noverlap,
    scaling="density",
    mode="magnitude",
)

plt.figure()
plt.pcolormesh(t_s, f_s, Sxx, shading="gouraud")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("Spectrogram (magnitude)")
plt.ylim(0, FS_HZ/2)
plt.colorbar(label="Magnitude")
plt.show()


### Troubleshooting (quick)
- **Channel error**: confirm `DEVICE` and `CHANNEL` (NI MAX → Devices and Interfaces).
- **Flat line / noisy signal**: check wiring, terminal config (RSE vs DIFF), and input range.
- **Timeout**: lower `FS_HZ`, reduce `N_SAMPLES`, or increase the `timeout` in the read call.
